# Трек B: VLM-счётчик на всех 4964 строках, два шарда на 2×T4 (шаг 2 задачи C3)
Вход: `ts-code`, `ts-models`, данные и веса Qwen-VL. GPU включён, интернет выключен. Возобновление: подключите вывод прошлой версии и скопируйте `artifacts/rowfeats/_parts` в `/kaggle/working/artifacts/rowfeats/_parts`.

In [ ]:
# Подготовка: код из датасета ts-code, данные соревнования, веса и колёса из ts-models. Интернет выключен.
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
code = [p.parent for p in INPUT.rglob('CONTRACT.md') if (p.parent / 'ts' / '__init__.py').is_file()]
data = [p.parent for p in INPUT.rglob('train_labels.csv')
        if (p.parent / 'tables.csv').is_file() and (p.parent / 'data_card.json').is_file()]
models = [p.parent for p in INPUT.rglob('yolo11x.pt')]
assert len(code) == 1, f'Подключите датасет ts-code (найдено {len(code)})'
assert len(data) == 1, f'Подключите данные соревнования (найдено {len(data)})'
assert len(models) == 1, f'Подключите датасет ts-models (найдено {len(models)})'
REPO, MODELS = Path('/kaggle/working/table-state'), models[0]
shutil.copytree(code[0], REPO, dirs_exist_ok=True)
os.environ['TS_DATA'] = str(data[0])
os.environ['TS_ARTIFACTS'] = '/kaggle/working/artifacts'
os.environ['YOLO_OFFLINE'] = 'true'
os.chdir(REPO)

# Колёса ставим без сети и без зависимостей: torch, torchvision и opencv уже есть в образе.
wheels = sorted((MODELS / 'wheels').glob('*.whl'))
need = [w for w in wheels if not (w.name.startswith('nvidia_ml_py') and importlib.util.find_spec('pynvml'))]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', *map(str, need)], check=True)
import torch, ultralytics
print('код:', (REPO / 'GIT_SHA').read_text().strip(), '| данные:', data[0], '| веса:', MODELS)
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ, включите GPU в настройках')


In [ ]:
QWEN = sorted(p.parent for p in INPUT.rglob('config.json') if 'qwen' in str(p).lower())
assert QWEN, 'Подключите веса Qwen-VL (Kaggle Models или датасет)'
QWEN = QWEN[0]
import torch
print('веса VLM:', QWEN, '| GPU:', torch.cuda.device_count())

In [ ]:
!python tracks/det/vlm/run_vlm.py --model {QWEN} --subset all --shard 0 --of 2 --device cuda:0 > /kaggle/working/shard0.log 2>&1 & python tracks/det/vlm/run_vlm.py --model {QWEN} --subset all --shard 1 --of 2 --device cuda:1; wait; tail -3 /kaggle/working/shard0.log

In [ ]:
!python tracks/det/vlm/run_vlm.py --subset all --of 2 --merge

In [ ]:
!ls -la /kaggle/working/artifacts/rowfeats && rm -rf /kaggle/working/table-state